<style>
.info {border-left:5px solid #2563eb;background:#eff6ff;padding:12px 16px;margin:12px 0}
.task {border-left:5px solid #d97706;background:#fffbeb;padding:12px 16px;margin:12px 0}
.success {border-left:5px solid #059669;background:#ecfdf5;padding:12px 16px;margin:12px 0}
.warning {border-left:5px solid #dc2626;background:#fef2f2;padding:12px 16px;margin:12px 0}
.definition {border-left:5px solid #7c3aed;background:#f5f3ff;padding:12px 16px;margin:12px 0}
</style>

        # Python for Data Science
        ## TW10 · Session 1 · Importing Text, CSV, and Excel Data

        **Course level:** developing beginner  
        **Prior learning:** files, pandas DataFrames, data types, and missing-value checks  
        **Duration:** approximately 90 minutes  
        **Method:** explain → predict → run → change → practise

        Importing is the first reproducibility test: another person should know which file was read, with which options, and what structure arrived.

        **Student name:** __________________________ &nbsp;&nbsp; **Date:** __________________


        ## Learning outcomes

        By the end of this session, you should be able to:

        - construct portable paths with pathlib
- read plain text with an explicit encoding
- import CSV data with selected columns and types
- read one or more Excel worksheets
- validate row counts, columns, types, and keys immediately after import


## Lesson plan

| Part | Time | Activity |
|---|---:|---|
| Retrieval | 10 min | Recall earlier Python and data skills |
| New concepts | 30 min | Learn with short, explained examples |
| Guided analysis | 20 min | Build one complete example with the lecturer |
| Student coding | 25 min | Complete five marked tasks |
| Review | 5 min | Knowledge check and exit ticket |

<div class="info"><strong>How to work:</strong> read the explanation, predict the result, run the cell with <code>Shift + Enter</code>, then change one thing. Edit only cells marked <strong>Student task</strong> unless your lecturer says otherwise.</div>


### Locate the supplied data folder

Portable paths use pathlib rather than manually joining Windows or Unix separators.


In [ ]:
from pathlib import Path
import pandas as pd

_candidates = [
    Path.cwd() / "data",
    Path.cwd() / "Python_Data_Science_TW09_TW17" / "data",
]
DATA_DIR = next((path for path in _candidates if path.exists()), _candidates[0])
if not DATA_DIR.exists():
    raise FileNotFoundError("Data folder not found. Start JupyterLab from the extracted course folder.")
print("Data folder:", DATA_DIR.resolve())


### List files before choosing one

A directory listing helps catch misspellings and prevents silently opening a similarly named file.


In [ ]:
available_files = sorted(path.name for path in DATA_DIR.iterdir())
for filename in available_files:
    print(filename)


## 1. Read plain text and understand encoding

An encoding maps bytes to characters. UTF-8 is a strong default, but older files may use Latin-1 or another encoding. Record the chosen encoding rather than relying on a machine-specific default.


### Read UTF-8 text

The context manager closes the file automatically. The result is one Python string.


In [ ]:
notes_path = DATA_DIR / "sample_notes_utf8.txt"
with notes_path.open("r", encoding="utf-8") as handle:
    notes_text = handle.read()

print(notes_text)
print("Characters:", len(notes_text))


        ### Read a Latin-1 file

        The correct encoding preserves accented characters and the pound sign.

<div class="warning"><strong>Take care:</strong> Do not hide decoding problems with errors='ignore'; discarded characters can change names, codes, or meaning.</div>


In [ ]:
latin1_path = DATA_DIR / "sample_notes_latin1.txt"
latin1_text = latin1_path.read_text(encoding="latin-1")
print(latin1_text)


## 2. Import CSV data

CSV is plain text organised as delimited rows. Import decisions include delimiter, header row, missing-value markers, selected columns, types, and date parsing.


### Read a CSV and inspect immediately

Never continue to analysis without checking the first rows, shape, column names, and data types.


In [ ]:
sales_path = DATA_DIR / "sample_sales.csv"
sales = pd.read_csv(sales_path)

print(sales.head())
print("Shape:", sales.shape)
print("Columns:", sales.columns.tolist())
print(sales.dtypes)


### Control columns, types, and dates

Explicit options reduce memory and prevent identifiers being treated as measurements.


In [ ]:
selected_sales = pd.read_csv(
    sales_path,
    usecols=["order_id", "order_date", "region", "units", "unit_price"],
    dtype={"order_id": "string", "region": "category"},
    parse_dates=["order_date"],
)

print(selected_sales.head())
print(selected_sales.dtypes)


### Create import-time missing markers

A real file may use several tokens for missingness. The example shows the option without changing the supplied clean file.


In [ ]:
missing_markers = ["", "NA", "N/A", "not recorded", "-"]
sales_with_rules = pd.read_csv(
    sales_path,
    na_values=missing_markers,
    keep_default_na=True,
)
print(sales_with_rules.isna().sum())


## 3. Import Excel workbooks

An Excel file can contain multiple worksheets with different structures. Specify the sheet by name and inspect available names first.


### Inspect worksheet names

ExcelFile reads workbook metadata and can then parse chosen sheets.


In [ ]:
workbook_path = DATA_DIR / "sample_workbook.xlsx"
workbook = pd.ExcelFile(workbook_path)
print("Worksheets:", workbook.sheet_names)


### Read one sheet or all sheets

Passing sheet_name=None returns a dictionary from sheet name to DataFrame.


In [ ]:
sales_sheet = pd.read_excel(workbook_path, sheet_name="Sales")
all_sheets = pd.read_excel(workbook_path, sheet_name=None)

print("Sales shape:", sales_sheet.shape)
for sheet_name, frame in all_sheets.items():
    print(sheet_name, frame.shape)


## 4. Validate at the boundary

Treat import as a boundary between an external source and your analysis. Check assumptions immediately:

- file exists and is the expected source;
- required columns are present;
- row count is plausible;
- key is unique when required;
- types and missingness are understood.


### Write readable import assertions

Assertions stop the notebook early with a useful message when a required assumption fails.


In [ ]:
required_columns = {"order_id", "order_date", "region", "units", "unit_price"}
assert required_columns.issubset(selected_sales.columns), "Required columns are missing"
assert len(selected_sales) > 0, "The imported table is empty"
assert selected_sales["order_id"].is_unique, "order_id must be unique"
assert (selected_sales["units"] >= 0).all(), "Units cannot be negative"

print("Import validation passed.")


## Guided import: create a small sales report

We read only needed columns, parse dates, validate the structure, calculate revenue, and aggregate by region.


### Import, validate, calculate, and summarise

The transformation starts only after the boundary checks pass.


In [ ]:
report_source = pd.read_csv(
    sales_path,
    usecols=["order_id", "order_date", "region", "units", "unit_price"],
    parse_dates=["order_date"],
)
assert report_source["order_id"].is_unique
assert report_source[["units", "unit_price"]].notna().all().all()

report_source["revenue"] = report_source["units"] * report_source["unit_price"]
region_report = (
    report_source.groupby("region", as_index=False)
    .agg(orders=("order_id", "count"), revenue=("revenue", "sum"))
    .sort_values("revenue", ascending=False)
)
print(region_report)


## Student coding lab

Use the supplied data files. Every import must be followed by at least two checks.


        ### Student task 1: Build portable paths

        <div class="task"><strong>Your job:</strong> Create paths for sample_sales.csv and sample_workbook.xlsx. Print each name, suffix, and whether it exists.</div>

        **Check your work**

        - The slash operator creates both paths.
- Both paths report True for exists.
- The suffixes are .csv and .xlsx.


In [ ]:
# STUDENT TASK 1
csv_path = DATA_DIR
excel_path = DATA_DIR
print(csv_path)
print(excel_path)


        ### Student task 2: Read and inspect text

        <div class="task"><strong>Your job:</strong> Read the UTF-8 notes file, split it into lines, and print the line count and final line.</div>

        **Check your work**

        - The file is opened with UTF-8.
- Blank trailing text does not create a misleading result.
- The final content line is printed.


In [ ]:
# STUDENT TASK 2
notes = ""
lines = []
print("Line count:", len(lines))
print("Final line:", lines[-1] if lines else None)


        ### Student task 3: Import selected CSV columns

        <div class="task"><strong>Your job:</strong> Read order_id, region, units, and unit_price. Keep order_id as string, then print shape, types, and missing counts.</div>

        **Check your work**

        - Only four columns are imported.
- order_id is a string dtype.
- Three structural checks are displayed.


In [ ]:
# STUDENT TASK 3
task_sales = pd.DataFrame()
print("Shape:", task_sales.shape)
print(task_sales.dtypes)
print(task_sales.isna().sum())


        ### Student task 4: Read and combine Excel sheets

        <div class="task"><strong>Your job:</strong> Read Sales and Regions, then left-merge the manager and target onto every sales row using region.</div>

        **Check your work**

        - Both sheets are read by name.
- All sales rows remain.
- The merge uses validate='many_to_one'.


In [ ]:
# STUDENT TASK 4
task_sales_sheet = pd.DataFrame()
task_regions_sheet = pd.DataFrame()
combined_excel = pd.DataFrame()
print(combined_excel)


        ### Student task 5: Create an import contract

        <div class="task"><strong>Your job:</strong> Import sample_sales.csv and assert its expected 8 rows, required columns, unique order ID, and positive units and prices.</div>

        **Check your work**

        - All four expectations are asserted.
- Messages explain a failed assumption.
- The valid supplied file reaches the final print.



**Optional extension:** Change one expected value deliberately, observe the failure, and restore it.


In [ ]:
# STUDENT TASK 5
contract_data = pd.read_csv(DATA_DIR / "sample_sales.csv")
# Add four assertions.
print("Rows:", len(contract_data))


        ## Knowledge check

        Answer these without running new code first.

        1. Why is pathlib preferable to manually joining path strings?
2. What does a character encoding control?
3. Why can dtype be important for an identifier?
4. How do you inspect workbook sheet names?
5. Why validate immediately after import?


## Exit ticket and preparation

<div class="success"><strong>Exit ticket:</strong> write one idea you can explain, one operation you can code, and one question that remains.</div>

**Before the next session:** Session 2 imports JSON, pickle, gzip, and ZIP formats and compares their risks and structures.
